# 📈 仮想通貨 次の足 上昇/下落 予測ツール
**データ**: CoinGecko API（無料・APIキー不要）  
**予測**: ランダムフォレスト（機械学習）  
**使い方**: 上のセルから順番に `Shift + Enter` で実行するだけ！

## ① ライブラリのインストール（初回のみ）

## ② インポート

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ---- ダークテーマ ----
plt.rcParams['figure.facecolor']  = '#0d1117'
plt.rcParams['axes.facecolor']    = '#161b22'
plt.rcParams['axes.edgecolor']    = '#30363d'
plt.rcParams['axes.labelcolor']   = '#c9d1d9'
plt.rcParams['xtick.color']       = '#8b949e'
plt.rcParams['ytick.color']       = '#8b949e'
plt.rcParams['text.color']        = '#c9d1d9'
plt.rcParams['grid.color']        = '#21262d'
plt.rcParams['grid.linewidth']    = 0.5
plt.rcParams['legend.facecolor']  = '#1c2128'
plt.rcParams['legend.edgecolor']  = '#30363d'

print('✅ インポート完了')

✅ インポート完了


## ③ 設定 ← **ここだけ変更すればOK**

In [2]:
# ==================================================
#  通貨を選ぶ（1つだけコメントを外す）
# ==================================================
COIN_ID   = 'bitcoin'      # ビットコイン
# COIN_ID = 'ethereum'     # イーサリアム
# COIN_ID = 'solana'       # ソラナ
# COIN_ID = 'binancecoin'  # BNB
# COIN_ID = 'ripple'       # XRP

# 学習に使う過去データの日数（多いほど精度が上がりやすい）
DAYS = 60

# ==================================================
print(f'設定完了 → 通貨: {COIN_ID} / 学習期間: {DAYS}日')

設定完了 → 通貨: bitcoin / 学習期間: 60日


## ④ データ取得

In [3]:
def fetch_ohlc(coin_id, days):
    """CoinGecko から OHLC（始値/高値/安値/終値）を取得"""
    url = f'https://api.coingecko.com/api/v3/coins/{coin_id}/ohlc'
    r = requests.get(url, params={'vs_currency': 'usd', 'days': days}, timeout=15)
    r.raise_for_status()
    df = pd.DataFrame(r.json(), columns=['ts', 'open', 'high', 'low', 'close'])
    df['ts'] = pd.to_datetime(df['ts'], unit='ms')
    df.set_index('ts', inplace=True)
    return df.sort_index()

def fetch_current(coin_id):
    """現在価格を取得"""
    r = requests.get(
        'https://api.coingecko.com/api/v3/simple/price',
        params={
            'ids': coin_id,
            'vs_currencies': 'usd',
            'include_24hr_change': 'true',
            'include_market_cap': 'true'
        }, timeout=10
    )
    r.raise_for_status()
    return r.json().get(coin_id, {})

print('📡 データ取得中...')
df = fetch_ohlc(COIN_ID, DAYS)
current = fetch_current(COIN_ID)

price  = current.get('usd', 0)
chg24  = current.get('usd_24h_change', 0)
mcap   = current.get('usd_market_cap', 0)
arrow  = '▲' if chg24 >= 0 else '▼'

print(f'\n{'='*45}')
print(f'  {COIN_ID.upper()} / USD')
print(f'  現在価格 : ${price:,.2f}')
print(f'  24H変動  : {arrow} {abs(chg24):.2f}%')
print(f'  時価総額 : ${mcap/1e9:.1f}B')
print(f'  取得件数 : {len(df)}本')
print(f'  期間     : {df.index[0].strftime("%Y/%m/%d")} 〜 {df.index[-1].strftime("%Y/%m/%d")}')
print(f'{'='*45}')

SyntaxError: f-string: expecting '}' (3936861588.py, line 34)

## ⑤ テクニカル指標の計算 & 機械学習

In [ ]:
def build_features(df):
    """テクニカル指標を特徴量として計算する"""
    d = df.copy()

    # 移動平均
    for w in [3, 5, 10, 20]:
        d[f'ma{w}'] = d['close'].rolling(w).mean()

    # モメンタム（価格変化率）
    for w in [1, 3, 5]:
        d[f'mom{w}'] = d['close'].pct_change(w)

    # ボラティリティ
    d['vol5']  = d['close'].pct_change().rolling(5).std()
    d['vol10'] = d['close'].pct_change().rolling(10).std()

    # RSI
    delta = d['close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['rsi'] = 100 - 100 / (1 + gain / (loss + 1e-9))

    # ボリンジャーバンド位置
    mid = d['close'].rolling(20).mean()
    std = d['close'].rolling(20).std()
    d['bb_pos'] = (d['close'] - mid) / (2 * std + 1e-9)

    # MACD
    ema12 = d['close'].ewm(span=12, adjust=False).mean()
    ema26 = d['close'].ewm(span=26, adjust=False).mean()
    d['macd']        = ema12 - ema26
    d['macd_signal'] = d['macd'].ewm(span=9, adjust=False).mean()
    d['macd_diff']   = d['macd'] - d['macd_signal']

    # 高値・安値レンジ比
    d['hl_ratio'] = (d['high'] - d['low']) / (d['close'] + 1e-9)

    # 目的変数: 次の足が上昇なら1
    d['target'] = (d['close'].shift(-1) > d['close']).astype(int)

    return d


FEATURES = [
    'ma3','ma5','ma10','ma20',
    'mom1','mom3','mom5',
    'vol5','vol10',
    'rsi','bb_pos',
    'macd','macd_signal','macd_diff',
    'hl_ratio'
]

# ---- 特徴量を計算 ----
feat_df = build_features(df)
data    = feat_df.dropna()

X = data[FEATURES].values
y = data['target'].values

# ---- 標準化 ----
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# ---- 学習/テスト分割（時系列なので末尾をテストに）----
X_tr, X_te, y_tr, y_te = train_test_split(X_sc, y, test_size=0.2, shuffle=False)

# ---- ランダムフォレストで学習 ----
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_tr, y_tr)

# ---- 精度評価 ----
y_pred = clf.predict(X_te)
acc    = accuracy_score(y_te, y_pred)

print('✅ 学習完了')
print(f'\nバックテスト精度: {acc*100:.1f}%')
print(f'テストデータ件数: {len(y_te)}本')
print()
print(classification_report(y_te, y_pred, target_names=['下落','上昇']))

## ⑥ 🎯 予測結果

In [ ]:
# 最新データで予測
latest    = data[FEATURES].iloc[[-1]].values
latest_sc = scaler.transform(latest)
pred      = clf.predict(latest_sc)[0]
prob      = clf.predict_proba(latest_sc)[0]   # [P(下落), P(上昇)]
prob_up   = prob[1]
prob_dn   = prob[0]
confidence = max(prob_up, prob_dn) * 100

# ---- 表示 ----
line = '=' * 50
print(line)
print(f'  予測結果  [{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}]')
print(line)
print(f'  通貨         : {COIN_ID.upper()} / USD')
print(f'  現在価格     : ${price:,.2f}  ({arrow}{abs(chg24):.2f}%)')
print()

if pred == 1:
    print('  次の足の予測 : 📈 上昇')
else:
    print('  次の足の予測 : 📉 下落')

print(f'  上昇確率     : {prob_up*100:.1f}%')
print(f'  下落確率     : {prob_dn*100:.1f}%')
print(f'  モデル信頼度 : {confidence:.1f}%')
print(f'  バックテスト精度: {acc*100:.1f}%')
print(line)

# 確率バー（テキストで可視化）
bar_len = 30
up_bar  = int(prob_up * bar_len)
dn_bar  = bar_len - up_bar
print(f'\n  上昇 [{"█"*up_bar}{"░"*dn_bar}] {prob_up*100:.1f}%')
print(f'  下落 [{"█"*dn_bar}{"░"*up_bar}] {prob_dn*100:.1f}%')
print()
print('  ⚠️  このシグナルは参考情報です。投資判断は自己責任でお願いします。')

## ⑦ チャート表示

In [ ]:
GREEN  = '#3fb950'
RED    = '#f85149'
BLUE   = '#58a6ff'
AMBER  = '#e3b341'
PURPLE = '#bc8cff'
GRAY   = '#8b949e'

fig = plt.figure(figsize=(15, 12))
pred_color = GREEN if pred == 1 else RED
pred_label = '📈 上昇予測' if pred == 1 else '📉 下落予測'
fig.suptitle(
    f'{COIN_ID.upper()}/USD  ${price:,.2f}  |  {pred_label}  |  信頼度 {confidence:.1f}%',
    fontsize=13, fontweight='bold', color=pred_color, y=0.98
)

gs  = GridSpec(4, 2, figure=fig, height_ratios=[3, 1.2, 1.2, 1.2], hspace=0.08, wspace=0.25)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :], sharex=ax1)
ax3 = fig.add_subplot(gs[2, :], sharex=ax1)
ax4 = fig.add_subplot(gs[3, 0])
ax5 = fig.add_subplot(gs[3, 1])

# ── 価格 + MA + BB ──
ax1.plot(feat_df.index, feat_df['close'], color=BLUE, linewidth=1.3, label='終値', zorder=5)
ax1.plot(feat_df.index, feat_df['ma5'],   color=GREEN,  linewidth=0.9, label='MA5', alpha=0.85)
ax1.plot(feat_df.index, feat_df['ma20'],  color=AMBER,  linewidth=0.9, label='MA20', alpha=0.85)

# ボリンジャーバンド再計算（表示用）
bb_mid = feat_df['close'].rolling(20).mean()
bb_std = feat_df['close'].rolling(20).std()
bb_u   = bb_mid + 2 * bb_std
bb_l   = bb_mid - 2 * bb_std
ax1.plot(feat_df.index, bb_u, color=GRAY, linewidth=0.7, linestyle='--', alpha=0.5, label='BB')
ax1.plot(feat_df.index, bb_l, color=GRAY, linewidth=0.7, linestyle='--', alpha=0.5)
ax1.fill_between(feat_df.index, bb_u, bb_l, color=GRAY, alpha=0.06)

# 最新価格マーカー
ax1.scatter(feat_df.index[-1], feat_df['close'].iloc[-1],
            color=pred_color, s=80, zorder=10, label=f'現在 {pred_label}')
ax1.set_ylabel('Price (USD)', fontsize=10)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(loc='upper left', fontsize=8, framealpha=0.4, ncol=6)
ax1.grid(True)

# ── RSI ──
ax2.plot(feat_df.index, feat_df['rsi'], color=BLUE, linewidth=1.0)
ax2.axhline(70, color=RED,   linewidth=0.8, linestyle='--', alpha=0.7)
ax2.axhline(30, color=GREEN, linewidth=0.8, linestyle='--', alpha=0.7)
ax2.axhline(50, color=GRAY,  linewidth=0.5, linestyle=':', alpha=0.5)
ax2.fill_between(feat_df.index, feat_df['rsi'], 70,
                 where=(feat_df['rsi'] > 70), color=RED,   alpha=0.12)
ax2.fill_between(feat_df.index, feat_df['rsi'], 30,
                 where=(feat_df['rsi'] < 30), color=GREEN, alpha=0.12)
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI (14)', fontsize=9)
ax2.grid(True)

# ── MACD ──
ax3.plot(feat_df.index, feat_df['macd'],        color=BLUE,  linewidth=1.0, label='MACD')
ax3.plot(feat_df.index, feat_df['macd_signal'], color=AMBER, linewidth=1.0, label='Signal')
hist_colors = feat_df['macd_diff'].apply(lambda v: GREEN if v >= 0 else RED)
ax3.bar(feat_df.index, feat_df['macd_diff'], color=hist_colors, alpha=0.55, width=0.03, label='Hist')
ax3.axhline(0, color=GRAY, linewidth=0.5)
ax3.set_ylabel('MACD', fontsize=9)
ax3.legend(loc='upper left', fontsize=7, framealpha=0.3, ncol=3)
ax3.grid(True)

# X軸を最後のサブプロットだけ表示
for ax in [ax1, ax2]:
    plt.setp(ax.get_xticklabels(), visible=False)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax3.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.setp(ax3.get_xticklabels(), rotation=30, ha='right', fontsize=8)

# ── 予測確率パイチャート ──
ax4.pie(
    [prob_up, prob_dn],
    labels=['上昇', '下落'],
    colors=[GREEN, RED],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 11},
    wedgeprops={'linewidth': 1, 'edgecolor': '#0d1117'}
)
ax4.set_title('予測確率', fontsize=11, pad=10)

# ── 特徴量の重要度 ──
importance = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
bars = ax5.barh(importance.index, importance.values,
                color=[BLUE if v > importance.median() else GRAY for v in importance.values],
                height=0.7)
ax5.set_title('特徴量の重要度', fontsize=11)
ax5.set_xlabel('Importance', fontsize=9)
ax5.grid(True, axis='x')

plt.savefig('crypto_prediction.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('✅ チャートを crypto_prediction.png に保存しました')

## ⑧ 再予測（このセルだけ繰り返し実行するだけ）

In [ ]:
# 最新データを再取得して予測を更新する
print('📡 最新データを取得中...')
df      = fetch_ohlc(COIN_ID, DAYS)
current = fetch_current(COIN_ID)
price   = current.get('usd', 0)
chg24   = current.get('usd_24h_change', 0)
arrow   = '▲' if chg24 >= 0 else '▼'

feat_df   = build_features(df)
data      = feat_df.dropna()
X         = scaler.transform(data[FEATURES].values)
latest_sc = scaler.transform(data[FEATURES].iloc[[-1]].values)
pred      = clf.predict(latest_sc)[0]
prob      = clf.predict_proba(latest_sc)[0]
prob_up   = prob[1]
prob_dn   = prob[0]
confidence= max(prob_up, prob_dn) * 100

line = '=' * 50
print(line)
print(f'  最新予測  [{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}]')
print(line)
print(f'  現在価格     : ${price:,.2f}  ({arrow}{abs(chg24):.2f}%)')
if pred == 1:
    print('  次の足の予測 : 📈 上昇')
else:
    print('  次の足の予測 : 📉 下落')
print(f'  上昇確率     : {prob_up*100:.1f}%')
print(f'  下落確率     : {prob_dn*100:.1f}%')
print(f'  モデル信頼度 : {confidence:.1f}%')
bar_len = 30
up_bar  = int(prob_up * bar_len)
dn_bar  = bar_len - up_bar
print(f'\n  上昇 [{"█"*up_bar}{"░"*dn_bar}] {prob_up*100:.1f}%')
print(f'  下落 [{"█"*dn_bar}{"░"*up_bar}] {prob_dn*100:.1f}%')
print(line)
print('  ⚠️  このシグナルは参考情報です。投資判断は自己責任でお願いします。')

---
### 📖 使い方まとめ
| セル | やること |
|------|----------|
| ① | 初回のみ実行（インストール）|
| ②③ | 毎回実行（設定・準備）|
| ④⑤ | データ取得・学習 |
| ⑥ | 予測結果を確認 |
| ⑦ | チャートを表示 |
| **⑧** | **最新予測だけ更新したいときは⑧だけ実行** |

### 通貨の変え方
セル③の `COIN_ID` を変えて、②から順に再実行するだけです。

### ⚠️ 注意
- 予測は統計的な参考値です。実際の投資判断には使用しないでください。
- CoinGecko無料APIにはレート制限があります（連続実行時は少し待ってください）。